### Shape Hierarchy

In [2]:
class Shape:

    def area(self):
        raise NotImplementedError("Subclasses must implement area")

    def perimeter(self):
        raise NotImplementedError("Subclasses must implement perimeter")


class Rectangle(Shape):

    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


class Square(Rectangle):

    def __init__(self, side):
        super().__init__(side, side)

In [3]:
rect = Rectangle(10, 5)
square = Square(4)

print("Rectangle Area:", rect.area())
print("Rectangle Perimeter:", rect.perimeter())

print("Square Area:", square.area())
print("Square Perimeter:", square.perimeter())

Rectangle Area: 50
Rectangle Perimeter: 30
Square Area: 16
Square Perimeter: 16


### Refactoring the Shape Hierarchy

In [4]:
class LoggerMixin:

    def log(self, message):
        print(f"[LOG]: {message}")


class RectangleDimensions:

    def __init__(self, width, height):
        self.width = width
        self.height = height


class SquareDimensions:

    def __init__(self, side):
        self.side = side


class Shape(LoggerMixin):

    def __init__(self, dimensions):
        self.dimensions = dimensions


class Rectangle(Shape):

    def area(self):
        area = self.dimensions.width * self.dimensions.height
        self.log(f"Rectangle area calculated: {area}")
        return area

    def perimeter(self):
        p = 2 * (self.dimensions.width + self.dimensions.height)
        self.log(f"Rectangle perimeter calculated: {p}")
        return p


class Square(Shape):

    def area(self):
        area = self.dimensions.side ** 2
        self.log(f"Square area calculated: {area}")
        return area

    def perimeter(self):
        p = 4 * self.dimensions.side
        self.log(f"Square perimeter calculated: {p}")
        return p

In [5]:
rect_dims = RectangleDimensions(10, 5)
square_dims = SquareDimensions(4)

rectangle = Rectangle(rect_dims)
square = Square(square_dims)

print(rectangle.area())
print(rectangle.perimeter())

print(square.area())
print(square.perimeter())

[LOG]: Rectangle area calculated: 50
50
[LOG]: Rectangle perimeter calculated: 30
30
[LOG]: Square area calculated: 16
16
[LOG]: Square perimeter calculated: 16
16


### Abstract Repository Interface for In-memory and File-based storage

In [6]:
from abc import ABC, abstractmethod


class UserRepository(ABC):

    @abstractmethod
    def save(self, user):
        pass

    @abstractmethod
    def get(self, user_id):
        pass

    @abstractmethod
    def list(self):
        pass

In [9]:
class User:

    def __init__(self, user_id, name):
        self.id = user_id
        self.name = name

    def __repr__(self):
        return f"User(id={self.id}, name={self.name})"

In [10]:
class InMemoryUserRepository(UserRepository):

    def __init__(self):
        self.users = {}

    def save(self, user):
        self.users[user.id] = user

    def get(self, user_id):
        return self.users.get(user_id)

    def list(self):
        return list(self.users.values())

In [11]:
repo = InMemoryUserRepository()

repo.save(User(1, "Ali"))
repo.save(User(2, "Sara"))

print(repo.get(1))
print(repo.list())

User(id=1, name=Ali)
[User(id=1, name=Ali), User(id=2, name=Sara)]


In [12]:
import json 

class FileUserRepository(UserRepository):

    def __init__(self, filename="users.json"):
        self.filename = filename

    def save(self, user):

        users = self.list()

        users.append({
            "id": user.id,
            "name": user.name
        })

        with open(self.filename, "w") as f:
            json.dump(users, f)

    def get(self, user_id):

        users = self.list()

        for user in users:
            if user["id"] == user_id:
                return user

        return None

    def list(self):

        try:
            with open(self.filename, "r") as f:
                return json.load(f)

        except FileNotFoundError:
            return []

In [13]:
repo = FileUserRepository()

repo.save(User(1,"Ali"))
repo.save(User(2,"Sara"))

print(repo.get(1))
print(repo.list())

{'id': 1, 'name': 'Ali'}
[{'id': 1, 'name': 'Ali'}, {'id': 2, 'name': 'Sara'}]


### Refactor a monolithic script into SOLID-compliant classes

#### The Monolithic Script (Bad Design)
Imagine a script that creates a user, saves it and sends an email.

In [14]:
class UserManager:

    def create_user(self, name, email):

        user = {"name": name, "email": email}

        print("Saving user to database")

        print("Sending welcome email to", email)

        return user
    

manager = UserManager()

manager.create_user("Ali", "ali@gmail.com")

Saving user to database
Sending welcome email to ali@gmail.com


{'name': 'Ali', 'email': 'ali@gmail.com'}

#### Refactored code

In [15]:
class User:

    def __init__(self, name, email):
        self.name = name
        self.email = email


class UserRepository:

    def save(self, user):
        print("Saving user to database")


class EmailService:

    def send_welcome_email(self, user):
        print("Sending welcome email to", user.email)


class UserService:

    def __init__(self, repository, email_service):

        self.repository = repository
        self.email_service = email_service


    def create_user(self, name, email):

        user = User(name, email)

        self.repository.save(user)

        self.email_service.send_welcome_email(user)

        return user
    


repo = UserRepository()

email_service = EmailService()

service = UserService(repo, email_service)

service.create_user("Ali", "ali@gmail.com")

Saving user to database
Sending welcome email to ali@gmail.com


### Notification System Using Strategy Pattern (Email, SMS, push)

In [16]:
from abc import ABC, abstractmethod


class NotificationStrategy(ABC):

    @abstractmethod
    def send(self, user, message):
        pass


class EmailNotification(NotificationStrategy):

    def send(self, user, message):
        print(f"Sending EMAIL to {user}: {message}")


class SMSNotification(NotificationStrategy):

    def send(self, user, message):
        print(f"Sending SMS to {user}: {message}")


class PushNotification(NotificationStrategy):

    def send(self, user, message):
        print(f"Sending PUSH notification to {user}: {message}")


class NotificationService:

    def __init__(self, strategy: NotificationStrategy):
        self.strategy = strategy

    def notify(self, user, message):
        self.strategy.send(user, message)


email_service = NotificationService(EmailNotification())
email_service.notify("Ali", "Welcome!")

sms_service = NotificationService(SMSNotification())
sms_service.notify("Ali", "OTP: 1234")

push_service = NotificationService(PushNotification())
push_service.notify("Ali", "You have a new message")

Sending EMAIL to Ali: Welcome!
Sending SMS to Ali: OTP: 1234
Sending PUSH notification to Ali: You have a new message
